# Interior Design Image Transformation Tester

This notebook allows you to test the image transformation feature of the Interior Design application. Upload a room image, provide your design preferences, and see the AI-powered transformation in action.

**Features:**
- Upload room image
- Input personalized design preferences
- Generate design brief
- Apply image transformation
- Compare original and transformed images side-by-side

In [1]:
import os
import json
import base64
from pathlib import Path
from io import BytesIO
from datetime import datetime
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import Slider
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
import sys

# Add the app directory to path to import our modules
sys.path.insert(0, '/workspaces/interior-design-site')

print("✓ All libraries imported successfully!")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Configuration for Image Transformation
CONFIG = {
    "output_dir": Path("generated_images"),
    "prompt_history_dir": Path("prompt_history"),
    "image_size": (768, 576),  # Width, Height (multiples of 64 for stable diffusion)
    "transformation_steps": 30,
    "guidance_scale": 7.5,
}

# Create directories
CONFIG["output_dir"].mkdir(exist_ok=True)
CONFIG["prompt_history_dir"].mkdir(exist_ok=True)

def build_detailed_prompt(client_data, theme_info):
    """Build detailed transformation prompt from application"""
    
    prompt = f"""
Transform this room into a {theme_info.get('theme_name', 'Modern')} interior.

STYLE:
{theme_info.get('style_description', 'Contemporary and functional')}

COLOR PALETTE:
{theme_info.get('color_palette', 'Warm neutrals and accent colors')}

MOOD:
Cozy, organized, welcoming, {theme_info.get('mood', 'comfortable')}

DESIGN ELEMENTS:
{theme_info.get('design_elements', 'Minimalist furniture, plants, natural textures')}

RULES:
- Preserve room architecture and layout
- Maintain realistic furniture scale
- Declutter space, organized storage solutions
- Warm interior lighting
- Natural materials and textures where appropriate
- Accessible and functional layout

PERSONALIZATION:
Hobbies & Interests: {client_data.get('hobbies', '')}
Space Requirements: {client_data.get('requirements', '')}
Client Preferences: {client_data.get('likes', '')}
Avoid: {client_data.get('dislikes', '')}

QUALITY SPECIFICATIONS:
- Architectural visualization quality
- Professional interior design photography
- Realistic perspective and proportions
- Coherent lighting and shadows
- High resolution details
"""
    return prompt.strip()


def build_transformation_prompt(full_prompt):
    """Build the transformation command prompt"""
    return f"""
Transform this room into: {full_prompt}

preserve room layout,
correct perspective,
realistic interior photography,
organized clutter-free space,
professional quality
"""

print("✓ Configuration and prompt builders set up successfully!")

In [ ]:
# Store uploaded file globally
uploaded_image = None
uploaded_image_path = None

# Create upload widget
file_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image:',
    layout=widgets.Layout(width='400px')
)

# Create client information form widgets
name_input = widgets.Text(
    value='Design Client',
    description='Name:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

about_input = widgets.Textarea(
    value='Professional who values functionality and aesthetics',
    description='About:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='80px')
)

preferred_colors_input = widgets.Text(
    value='Warm earth tones, sage green, cream',
    description='Preferred Colors:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

likes_input = widgets.Textarea(
    value='Modern minimalist, natural light, open spaces, plants',
    description='Likes:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='80px')
)

dislikes_input = widgets.Textarea(
    value='Clutter, bright colors, heavy furniture',
    description='Dislikes:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='80px')
)

hobbies_input = widgets.Text(
    value='Reading, yoga, cooking, gardening',
    description='Hobbies:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

requirements_input = widgets.Textarea(
    value='Good workspace for remote work, storage for books, meditation area',
    description='Requirements:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='80px')
)

additional_comments = widgets.Textarea(
    value='Budget-friendly transformations preferred',
    description='Comments:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='60px')
)

# Display upload and form
display(HTML("<h4 style='color: #C9A86A;'>📸 Upload Room Image</h4>"))
display(file_upload)

display(HTML("<h4 style='color: #C9A86A;'>👤 Client Information</h4>"))
display(name_input, about_input, preferred_colors_input, likes_input, dislikes_input, hobbies_input, requirements_input, additional_comments)

print("✓ Upload interface created successfully!")

In [ ]:
def load_uploaded_image():
    """Load and display the uploaded image"""
    global uploaded_image, uploaded_image_path
    
    if not file_upload.value:
        print("❌ No image uploaded yet. Please upload an image first.")
        return False
    
    # Get the uploaded file
    uploaded_filename = list(file_upload.value.keys())[0]
    file_content = file_upload.value[uploaded_filename]['content']
    
    # Save to temporary file
    temp_path = Path('temp_room_image.jpg')
    with open(temp_path, 'wb') as f:
        f.write(file_content)
    
    # Load as PIL Image
    uploaded_image = Image.open(temp_path).convert('RGB')
    uploaded_image_path = str(temp_path)
    
    # Resize for processing
    original_size = uploaded_image.size
    uploaded_image = uploaded_image.resize(CONFIG["image_size"])
    
    print(f"✓ Image loaded successfully!")
    print(f"  Original size: {original_size}")
    print(f"  Processing size: {CONFIG['image_size']}")
    
    return True


def display_original_image():
    """Display the original uploaded image"""
    global uploaded_image
    
    if uploaded_image is None:
        print("❌ No image loaded. Run 'load_uploaded_image()' first.")
        return
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(uploaded_image)
    ax.set_title('📸 Original Room Image', fontsize=16, fontweight='bold', color='#C9A86A', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Image dimensions: {uploaded_image.size}")

# Run the loading and display
if load_uploaded_image():
    display_original_image()

In [ ]:
def generate_design_brief():
    """Generate design brief from client preferences"""
    
    # Collect client data from input fields
    client_data = {
        'name': name_input.value,
        'about': about_input.value,
        'preferred_colors': preferred_colors_input.value,
        'likes': likes_input.value,
        'dislikes': dislikes_input.value,
        'hobbies': hobbies_input.value,
        'requirements': requirements_input.value,
        'additional_comments': additional_comments.value
    }
    
    # Create theme from client data
    theme_info = {
        "theme_name": "Personalized Design",
        "style_description": f"Based on preference for {client_data.get('likes', 'modern style')}",
        "color_palette": client_data.get('preferred_colors', 'Neutral tones'),
        "mood": "Comfortable, welcoming, and functional",
        "design_elements": f"Incorporating hobbies: {client_data.get('hobbies', 'reading, relaxation')}"
    }
    
    return client_data, theme_info


def display_design_brief(client_data, theme_info):
    """Display the generated design brief"""
    
    full_prompt = build_detailed_prompt(client_data, theme_info)
    
    print("\n" + "="*80)
    print("🎨 PERSONALIZED DESIGN BRIEF".center(80))
    print("="*80)
    print(f"\nClient: {client_data['name']}")
    print(f"Theme: {theme_info['theme_name']}")
    print("\n" + "-"*80)
    print("DETAILED PROMPT:")
    print("-"*80)
    print(full_prompt)
    print("\n" + "="*80 + "\n")
    
    return full_prompt


# Generate and display design brief
client_data, theme_info = generate_design_brief()
full_prompt = display_design_brief(client_data, theme_info)

print("✓ Design brief generated successfully!")

In [ ]:
# Initialize transformer
transformer = None
transformed_image = None

def initialize_transformer():
    """Initialize the image transformer"""
    global transformer
    
    try:
        from image_service_hf_api import get_transformer
        transformer = get_transformer()
        print("✓ Image transformer initialized successfully!")
        print(f"  Device: {transformer.device}")
        return True
    except Exception as e:
        print(f"❌ Error initializing transformer: {str(e)}")
        print("\nTroubleshooting:")
        print("  1. Make sure you have installed required packages:")
        print("     pip install torch diffusers transformers pillow numpy")
        print("  2. Check that HF_API_TOKEN is set in .env file")
        print("  3. Ensure GPU/CUDA is available (optional but recommended)")
        return False


def apply_transformation():
    """Apply image transformation using the service"""
    global transformer, uploaded_image, transformed_image
    
    if uploaded_image is None:
        print("❌ No image loaded. Please upload an image first.")
        return False
    
    if transformer is None:
        print("❌ Transformer not initialized. Run 'initialize_transformer()' first.")
        return False
    
    try:
        print("\n🔄 Starting image transformation...")
        print("   This may take a few minutes depending on your hardware.\n")
        
        # Use the transformer's transform_room method
        result = transformer.transform_room(uploaded_image, client_data, theme_info)
        
        if result['status'] == 'success':
            # Load transformed image from result
            if 'image_base64' in result:
                # Decode base64 image
                img_data = base64.b64decode(result['image_base64'])
                transformed_image = Image.open(BytesIO(img_data))
            elif 'image_path' in result:
                # Load from file path
                transformed_image = Image.open(result['image_path'])
            
            print("✓ Transformation completed successfully!")
            print(f"  Theme: {result.get('theme', 'Custom')}")
            print(f"  Timestamp: {result.get('timestamp', 'N/A')}")
            if 'prompt_history' in result:
                print(f"  Prompt saved to: {result['prompt_history']}")
            
            return True
        else:
            print(f"❌ Transformation failed: {result.get('message', 'Unknown error')}")
            return False
            
    except Exception as e:
        print(f"❌ Error during transformation: {str(e)}")
        import traceback
        traceback.print_exc()
        return False


# Initialize and apply transformation
if initialize_transformer():
    apply_transformation()

In [ ]:
def display_transformed_image():
    """Display the transformed image"""
    global transformed_image
    
    if transformed_image is None:
        print("❌ No transformed image available. Run 'apply_transformation()' first.")
        return
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(transformed_image)
    ax.set_title('✨ Transformed Room Image', fontsize=16, fontweight='bold', color='#C9A86A', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Transformed image dimensions: {transformed_image.size}")


# Display results if transformation was successful
if transformed_image is not None:
    display_transformed_image()
else:
    print("⏳ Waiting for transformation results...")

In [ ]:
def display_comparison():
    """Display side-by-side comparison of original and transformed images"""
    global uploaded_image, transformed_image
    
    if uploaded_image is None or transformed_image is None:
        print("❌ Both images are required for comparison.")
        if uploaded_image is None:
            print("   - Original image not loaded")
        if transformed_image is None:
            print("   - Transformed image not available")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Original image
    axes[0].imshow(uploaded_image)
    axes[0].set_title('Original Room', fontsize=14, fontweight='bold', color='#555', pad=10)
    axes[0].axis('off')
    
    # Transformed image
    axes[1].imshow(transformed_image)
    axes[1].set_title('Transformed Room', fontsize=14, fontweight='bold', color='#C9A86A', pad=10)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Comparison displayed successfully!")


def save_comparison_image():
    """Save comparison image to file"""
    global uploaded_image, transformed_image
    
    if uploaded_image is None or transformed_image is None:
        print("❌ Both images required for saving comparison.")
        return False
    
    try:
        # Create side-by-side comparison
        width = uploaded_image.width + transformed_image.width + 20  # Add gap
        height = max(uploaded_image.height, transformed_image.height)
        
        comparison = Image.new('RGB', (width, height), color='white')
        comparison.paste(uploaded_image, (0, 0))
        comparison.paste(transformed_image, (uploaded_image.width + 20, 0))
        
        # Save comparison
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"comparison_{timestamp}.png"
        comparison.save(filename)
        
        print(f"✓ Comparison saved as: {filename}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving comparison: {str(e)}")
        return False


# Display comparison if both images are available
if uploaded_image is not None and transformed_image is not None:
    display_comparison()
    print("\n" + "="*80)
    
    # Save comparison
    if save_comparison_image():
        print("="*80)
else:
    print("⏳ Waiting for both original and transformed images...")

## 9. Summary and Troubleshooting Guide

### ✅ What This Notebook Does

1. **Upload Images**: Select a room image from your computer
2. **Collect Preferences**: Provide client design preferences
3. **Generate Brief**: Creates a detailed design brief based on preferences
4. **Transform**: Uses AI to transform the room according to the brief
5. **Compare**: Displays original and transformed images side-by-side

### 🎯 Workflow Summary

The notebook follows these steps:
1. Import required libraries
2. Set up prompts and configuration
3. Create upload interface for image and client data
4. Load and display the original image
5. Generate design brief from client data
6. Initialize the transformer service
7. Apply transformation to the image
8. Display transformation results
9. Compare original vs transformed images

### ❌ Troubleshooting

#### Issue: "No module named 'diffusers'"
**Solution**: Install required packages
```
pip install torch diffusers transformers pillow numpy ipywidgets
```

#### Issue: "HuggingFace API token not found"
**Solution**: Create a `.env` file with:
```
HF_API_TOKEN=your_huggingface_token_here
```

#### Issue: GPU/CUDA not available
**Solution**: The transformer will automatically fall back to CPU, but it will be slower.

#### Issue: Transformation takes too long
**Solution**: 
- GPU processing is much faster than CPU
- The first run downloads models (takes longer)
- Subsequent runs use cached models
- Reduce `num_inference_steps` in CONFIG for faster (lower quality) results

### 📝 Customizing Preferences

You can modify the default values in **Section 3** (Create Image Upload Interface):
- `name_input.value` - Client name
- `preferred_colors_input.value` - Color preferences
- `likes_input.value` - Things the client likes
- `dislikes_input.value` - Things the client dislikes
- `hobbies_input.value` - Client hobbies/interests
- `requirements_input.value` - Space requirements

### 🔄 Re-running the Transformation

To transform with different preferences:
1. Modify the input fields in **Section 3**
2. Re-run **Section 5** (Generate Design Brief)
3. Re-run **Section 6** (Apply Transformation)
4. Re-run **Section 8** (Compare Images)

## 8. Compare Original and Transformed Images (Side-by-Side)

## 7. Display Transformation Results

## 6. Apply Image Transformation

**Note:** This section will initialize and use the actual image transformation service from the application. Make sure you have the required dependencies installed (torch, diffusers, PIL, etc.)

## 5. Generate Design Brief and Prepare for Transformation

## 4. Load and Display Uploaded Image

## 3. Create Image Upload Interface

## 2. Set Up Configuration and Prompts

## 1. Import Required Libraries